In [ ]:
# Linalg libraries
import numpy as np

# Plotting libraries
import matplotlib.pyplot as plt

# Data management libraries
import h5py as hf
from dataclasses import dataclass
import glob

In [ ]:
@dataclass
class ModelTrajectory:
    # Model data
    width: int
    depth: int
    activation: str
    lr: float
    architecture: str
    ds_size: float

    # Loss data
    train_loss: np.ndarray
    train_loss_std: np.ndarray
    test_loss: np.ndarray
    test_loss_std: np.ndarray

    # Accuracy data
    train_accuracy: np.ndarray
    train_accuracy_std: np.ndarray
    test_accuracy: np.ndarray
    test_accuracy_std: np.ndarray

    # CV data
    entropy: np.ndarray
    entropy_std: np.ndarray
    trace: np.ndarray
    trace_std: np.ndarray

In [ ]:
def load_model_trajectory(file_root, file_signature, accuracy = True):
    # Glob all files with the correct signature
    files = glob.glob(file_root + '/*' + file_signature + '*.h5')

    # Extract model parameters
    model_params = files[0].split("/")[-1].split('_')
    width = int(model_params[2])
    depth = int(model_params[3])
    activation = model_params[4]
    ds_size = float(model_params[5])    
    lr = float(model_params[7])
    architecture = model_params[1]

    # Load data
    train_losses = []
    test_losses = []

    if accuracy:
        train_accuracies = []
        test_accuracies = []
    
    entropies = []
    traces = []

    for file in files:
        with hf.File(file, 'r') as f:
            if file.split("/")[-1].split("_")[0] == "train":
                train_losses.append(f['loss'][:])
                if accuracy:
                    train_accuracies.append(f['accuracy'][:])
                entropies.append(f['entropy'][:])
                traces.append(f['trace'][:])
            else:
                test_losses.append(f['loss'][:])
                if accuracy:
                    test_accuracies.append(f['accuracy'][:])

    # Take the means, std and build the ModelTrajectory object
    train_loss = np.mean(train_losses, axis=0)
    test_loss = np.mean(test_losses, axis=0)
    train_loss_std = np.std(train_losses, axis=0)
    test_loss_std = np.std(test_losses, axis=0)

    if accuracy:
        train_accuracy = np.mean(train_accuracies, axis=0)
        test_accuracy = np.mean(test_accuracies, axis=0)

        train_accuracy_std = np.std(train_accuracies, axis=0)
        test_accuracy_std = np.std(test_accuracies, axis=0)

    entropy = np.mean(entropies, axis=0)
    trace = np.mean(traces, axis=0)
    entropy_std = np.std(entropies, axis=0)
    trace_std = np.std(traces, axis=0)

    return ModelTrajectory(
        width=width,
        depth=depth,
        activation=activation,
        lr=lr,
        architecture=architecture,
        ds_size=ds_size,
        train_loss=train_loss,
        train_loss_std=train_loss_std,
        test_loss=test_loss,
        test_loss_std=test_loss_std,
        train_accuracy=train_accuracy,
        train_accuracy_std=train_accuracy_std,
        test_accuracy=test_accuracy,
        test_accuracy_std=test_accuracy_std,
        entropy=entropy,
        entropy_std=entropy_std,
        trace=trace,
        trace_std=trace_std
    )

def load_model_trajectories(root_path):
    files = glob.glob(root_path + '/*.h5')

    # Get unique names
    unique_names = []
    for file in files:
        unique_names.append(
            "_".join(file.split('/')[-1].split('_')[2:7])
            )

    unique = np.unique(unique_names)

    # Load data
    data = []
    for name in unique:
        data.append(load_model_trajectory(root_path, name))

    return data   

In [ ]:
perceptron_mse_order_two = load_model_trajectories("perceptron/mse-2")
perceptron_mse_order_four = load_model_trajectories("perceptron/mse-4")
# perceptron_ce = load_model_trajectories("perceptron/ce")

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(15, 15))

# # MSE order two plots
for item in perceptron_mse_order_two:
    # Plot loss curves and label with activation with loss as error bands
    ax[0, 0].plot(item.train_loss, label=item.activation)
    ax[0, 0].fill_between(
        np.arange(len(item.train_loss)),
        item.train_loss - item.train_loss_std,
        item.train_loss + item.train_loss_std,
        alpha=0.3
    )

    # Plot trace curves on second row with activation as labels and error bands
    ax[1, 0].plot(item.trace, label=item.activation)
    ax[1, 0].fill_between(
        np.arange(len(item.trace)),
        item.trace - item.trace_std,
        item.trace + item.trace_std,
        alpha=0.3
    )

    # Plot entropy curves on third row with activation as labels and error bands
    ax[2, 0].plot(item.entropy, label=item.activation)
    ax[2, 0].fill_between(
        np.arange(len(item.entropy)),
        item.entropy - item.entropy_std,
        item.entropy + item.entropy_std,
        alpha=0.3
    )

# # MSE order four plots
for item in perceptron_mse_order_four:
    # Plot loss curves and label with activation with loss as error bands
    ax[0, 1].plot(item.train_loss, label=item.activation)
    ax[0, 1].fill_between(
        np.arange(len(item.train_loss)),
        item.train_loss - item.train_loss_std,
        item.train_loss + item.train_loss_std,
        alpha=0.3
    )

    # Plot trace curves on second row with activation as labels and error bands
    ax[1, 1].plot(item.trace, label=item.activation)
    ax[1, 1].fill_between(
        np.arange(len(item.trace)),
        item.trace - item.trace_std,
        item.trace + item.trace_std,
        alpha=0.3
    )

    # Plot entropy curves on third row with activation as labels and error bands
    ax[2, 1].plot(item.entropy, label=item.activation)
    ax[2, 1].fill_between(
        np.arange(len(item.entropy)),
        item.entropy - item.entropy_std,
        item.entropy + item.entropy_std,
        alpha=0.3
    )

# Cross entropy plots
# for item in perceptron_ce:
#     # Plot loss curves and label with activation with loss as error bands
#     ax[0, 2].plot(item.train_accuracy, label=item.activation)
#     ax[0, 2].fill_between(
#         np.arange(len(item.train_accuracy)),
#         item.train_accuracy - item.train_loss_std,
#         item.train_accuracy + item.train_loss_std,
#         alpha=0.3
#     )

#     # Plot trace curves on second row with activation as labels and error bands
#     ax[1, 2].plot(item.trace, label=item.activation)
#     ax[1, 2].fill_between(
#         np.arange(len(item.trace)),
#         item.trace - item.trace_std,
#         item.trace + item.trace_std,
#         alpha=0.3
#     )

#     # Plot entropy curves on third row with activation as labels and error bands
#     ax[2, 2].plot(item.entropy, label=item.activation)
#     ax[2, 2].fill_between(
#         np.arange(len(item.entropy)),
#         item.entropy - item.entropy_std,
#         item.entropy + item.entropy_std,
#         alpha=0.3
#     )

# Set titles for different losses
ax[0, 0].set_title("MSE order 2")
ax[0, 1].set_title("MSE order 4")
ax[0, 2].set_title("Cross Entropy")

# set x labels on bottom row
ax[2, 0].set_xlabel("Epochs")
ax[2, 1].set_xlabel("Epochs")
ax[2, 2].set_xlabel("Epochs")

# Set y labels
ax[0, 0].set_ylabel("Loss")
ax[1, 0].set_ylabel("Trace")
ax[2, 0].set_ylabel("Entropy")

# Set legends
ax[0, 0].legend()
ax[0, 1].legend()
ax[0, 2].legend()

# Set trace plots to logx-logy
ax[0, 0].set_yscale('log')
ax[0, 1].set_yscale('log')
ax[0, 2].set_yscale('log')
# ax[1, 0].set_xscale('log')
# ax[1, 1].set_xscale('log')
# ax[1, 2].set_xscale('log')       


# plt.legend()
plt.show()

In [ ]:
fuel_mse_order_two = load_model_trajectories("fuel-ds/mse-2")
fuel_mse_order_four = load_model_trajectories("fuel-ds/mse-4")